# 02 · TorchDistributor — 1×1 GPU

학습 함수는 [`torch_distributor_trainer.py`](https://github.com/Aiden-Jeon/databricks-distributed-training/blob/main/02-script-based/torch_distributor_trainer.py)의 `train_fn(...)`입니다. 본 노트북은 `TorchDistributor(num_processes=1, local_mode=True).run(...)`로 driver 안에서 1개 child 프로세스를 띄워 학습합니다.

world_size=1에서 DDP all_reduce는 no-op이지만, launcher·`train_fn` 시그니처를 1×N·M×N과 일치시켜 토폴로지만 바꿔 확장할 수 있습니다.

> 한 노트북 세션에서 `TorchDistributor.run`을 연속 호출하면 driver py4j callback channel이 단절되는 현상이 관찰되어 토폴로지별로 노트북을 분리했습니다. 1×N은 [`03-launch_torch_distributor_1xN.ipynb`](03-launch_torch_distributor_1xN.ipynb), M×N은 [`04-launch_torch_distributor_MxN.ipynb`](04-launch_torch_distributor_MxN.ipynb)에서 다룹니다.

**클러스터**: Single node, DBR 17.3 LTS ML, Driver `g5.12xlarge` (1× A10G로도 충분), Workers 0, Autoscaling off.

In [ ]:
%run ./00-setup

## 학습 함수 import (cloudpickle 회피용 wrapper)

In [ ]:
import os
import sys

NOTEBOOK_PATH = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)
SCRIPT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)
print(f"SCRIPT_DIR={SCRIPT_DIR}")


def td_train_fn(**kwargs):
    """TorchDistributor child에 보낼 thin wrapper.

    노트북 셀에서 module-level `train_fn`을 import해 TD에 직접 넘기면 cloudpickle이
    module reference로 직렬화하고, child 프로세스의 fresh sys.path에는 SCRIPT_DIR이
    없어 `from torch_distributor_trainer import train_fn`이 실패한다. inline 함수는
    bytecode가 by-value로 pickling되므로, child에서 sys.path를 보강한 뒤 lazy import해
    위 문제를 회피한다."""
    import sys

    sd = kwargs.get("script_dir")
    if sd and sd not in sys.path:
        sys.path.insert(0, sd)
    from torch_distributor_trainer import train_fn

    return train_fn(**kwargs)

## 1×1 GPU

In [ ]:
import mlflow
from pyspark.ml.torch.distributor import TorchDistributor

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="td-1x1", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=1,
    local_mode=True,
    use_gpu=True,
).run(
    td_train_fn,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_path=os.path.join(CKPT_DIR, "td-1x1.pt"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    topology="1x1",
    script_dir=SCRIPT_DIR,
)